# RunPod GPU
Rent a GPU, run notebook cells on it from VS Code, then delete it. Run steps 1–5 in order.
**Normal kernel** = your `.venv`. **GPU kernel** = the RunPod link from step 1.

## 0 · GPUs and prices (optional, normal kernel)
GPUs you can rent right now, cheapest first. Copy a name into `GPU` in step 1. 24 GB is plenty for these notebooks.

In [ ]:
import json, subprocess

gpus = json.loads(subprocess.check_output(["runpodctl", "gpu", "list"]))
for g in sorted(gpus, key=lambda g: g.get("securePricePerHr") or 99):
    if g.get("securePricePerHr") and "NVIDIA" in g["gpuId"] and "MIG" not in g["gpuId"]:
        print(f'${g["securePricePerHr"]}/hr  {g["memoryInGb"]} GB  {g["gpuId"]}')

## 1 · Start a GPU (normal kernel)
Rents a GPU machine on RunPod with PyTorch and Jupyter ready, waits until it's up (1–3 min), then prints a **Notebook** link (step 2) and a **Terminal** command (step 2b).
**Charges start now** (A40 ≈ $0.59/hr). Error, e.g. sold out? Change `GPU` and run again.

In [ ]:
import json, secrets, subprocess

GPU = "NVIDIA A40"
pw = secrets.token_urlsafe(8)  # one-time Jupyter password
r = subprocess.run([
    "runpodctl", "pod", "create", "--name", "learn-gpu", "--template-id", "runpod-torch-v280",
    "--gpu-id", GPU, "--min-cuda-version", "12.8", "--env", json.dumps({"JUPYTER_PASSWORD": pw}), "--wait"],
    capture_output=True, text=True)
if r.returncode:
    print(r.stderr)
else:
    pod = json.loads(r.stdout)
    print("Notebook:", f"https://{pod['id']}-8888.proxy.runpod.net/?token={pw}")
    print("Terminal:", pod["ssh"]["ssh_command"], "-o StrictHostKeyChecking=no -o UserKnownHostsFile=/dev/null")

## 2 · Connect to the GPU (clicks)
Top-right **Select Kernel** → **Select Another Kernel…** → **Existing Jupyter Server…** → paste the link → Enter → Enter → **Python 3 (ipykernel)**.
From now on, cells run on the GPU. The same clicks work in any other notebook.

## 2b · Or work from your terminal (SSH)
Paste the **Terminal** line into your terminal. You're now on the GPU computer: everything you type runs there. `exit` to leave.

| | Notebook (GPU kernel) | Terminal (after SSH) |
|---|---|---|
| Go to your files | `%cd /workspace` | `cd /workspace` |
| Run a script | `%run train.py` | `python train.py` |
| Any command | `!nvidia-smi` | `nvidia-smi` |
| Delete the GPU | `!runpodctl remove pod $RUNPOD_POD_ID` | `runpodctl remove pod $RUNPOD_POD_ID` |

Same GPU, same files. The notebook just puts `%` or `!` in front.

**Move files** (folders too) — two terminals:
- To the GPU: your computer `runpodctl send myfile` → GPU (in `/workspace`) `runpodctl receive <code>`
- Back: GPU `runpodctl send model.pth` → your computer `runpodctl receive <code>`

## 3 · Check the GPU (GPU kernel)
Prints the GPU's name.

In [ ]:
import torch
print(torch.cuda.get_device_name(0))

## Example (optional, GPU kernel)
Moves data and a model to the GPU (`.to(device)`) and compares speed with the CPU.

In [ ]:
import time, torch

device = torch.device("cuda")
x = torch.randn(8000, 8000)                         # data on CPU
t = time.time(); x @ x; print(f"CPU: {time.time() - t:.2f} s")
x = x.to(device); x @ x; torch.cuda.synchronize()   # move to GPU (+ warm-up)
t = time.time(); x @ x; torch.cuda.synchronize(); print(f"GPU: {time.time() - t:.3f} s")
model = torch.nn.Linear(8000, 10).to(device)        # move a model to GPU
print(model(x).shape, next(model.parameters()).device)

## Files & scripts (GPU kernel)
The GPU is a different computer: cell code goes automatically, **files don't**. Run `%cd /workspace` first — files go there.

| To | How |
|---|---|
| Get your repo (push first) | GPU: `!git clone https://github.com/faisalanjum/LLM-fine-tuning` |
| Send a file / folder | Your terminal: `runpodctl send myfolder` → GPU: `!runpodctl receive <code>` |
| Run a script (`.py`) on the GPU | Send it (row above) → GPU: `%run train.py` |
| Get results back (before step 4) | GPU: `!runpodctl send model.pth` → Your terminal: `runpodctl receive <code>` |

Other GPU notebooks: first `%cd` into their folder, e.g. `%cd /workspace/LLM-fine-tuning/Training`.
Your terminal runs on your computer, not the GPU. Terminal on the GPU: open the link in a browser → File → New → Terminal.

## 4 · Delete the GPU (GPU kernel)
Deletes the machine; charges stop. Ignore the "config file not found" line. VS Code then says the kernel died — expected.

In [ ]:
!runpodctl remove pod $RUNPOD_POD_ID

## 5 · Confirm (switch back to normal kernel)
**Select Kernel** → **Select Another Kernel…** → **Python Environments…** → `.venv`, then run.
`[]` = nothing running, no charges. A machine still listed? Run `!runpodctl pod delete <id>`.

In [ ]:
!runpodctl pod list --all